# First Pass at Cleaning Lyft Data

Each CSV holds about one month of ridership data.

## Mount drive to access 255 Final Project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Requirements

In [ ]:
import os
import pandas as pd
import geopandas as gpd

# Data Cleaning Functions

In [ ]:
###
# This function will take a Lyft ridership data CSV (which holds ~1 month of
# ridership data) and return a dataframe with only rides that start, end, or both
# in Oakland or Berkeley.
#
# Parameters:
#   csv_file: string -> path to csv with ridership data
#
# Returns:
#   dataframe -> holds only Oakland/Berkeley rows of the input csv
###
def create_oak_berk_df(csv_file):
  # Read in csv and save as dataframe
  df = pd.read_csv(csv_file)
  # Create mask to filter only rides that either start, end, or both in Berkeley/Oakland
  oak_berk_start_mask = df['start_station_id'].str.startswith('BK') | df['start_station_id'].str.startswith('OK')
  oak_berk_end_mask = df['end_station_id'].str.startswith('BK') | df['end_station_id'].str.startswith('OK')
  oak_berk_mask = oak_berk_start_mask | oak_berk_end_mask
  # Apply mask to create dataframe with only rides in Berkeley/Oakland
  oak_berk_df = df.loc[oak_berk_mask]
  # Return filtered dataframe
  return oak_berk_df


In [ ]:
# This function will take a ridership dataframe and return the station counts
# for where rides start, and another for where rides end.
#
# Parameters:
#   df: pandas.Dataframe -> aggregate dataframe holding all data for
#       Oakland/Berkeley rides
#
# Returns:
#   stations_gdf: (pandas.GeoDataFrame) ->
#      id column holds station id
#      start_count column holds number of rides that start there
#      end_count holds number of rides that end there
#      combined_count is the sum of start_count and end_count
#      name
#      lat
#      long
#      geometry
#
def create_station_counts_gdf(df):
  # Filter only non-SF start stations to get value counts for starting stations
  # and filter only non-SF end stations to get value counts for ending stations.
  # Drop row if starting id or ending id is NaN.
  start_df = df.loc[
      df['start_station_id'].notna() &
      ~df['start_station_id'].fillna('').str.startswith("SF")
  ]
  end_df = df.loc[
      df['end_station_id'].notna() &
      ~df['end_station_id'].fillna('').str.startswith("SF")
  ]

  # Get value counts and combine into single dataframe
  start_counts = start_df['start_station_id'].value_counts()
  end_counts = end_df['end_station_id'].value_counts()
  station_counts = pd.DataFrame({
      "start_count": start_counts,
      "end_count": end_counts
  })
  station_counts = station_counts.reset_index().rename(columns={"index": "id"})
  # Fill in zero for missing values
  station_counts = station_counts.fillna(0)
  # Change from float to int
  station_counts["start_count"] = station_counts["start_count"].astype(int)
  station_counts["end_count"] = station_counts["end_count"].astype(int)

  # Add combined count column
  station_counts['combined_count'] = station_counts['start_count'] + station_counts['end_count']

  # Get station names
  station_names = pd.concat([
      df[["start_station_id", "start_station_name"]].rename(
          columns={"start_station_id": "id", "start_station_name": "name"}
      ),
      df[["end_station_id", "end_station_name"]].rename(
          columns={"end_station_id": "id", "end_station_name": "name"}
      )
  ]).dropna(subset=["id"]).drop_duplicates()
  print("station names length: ", len(station_names))


  # Get station locations
  station_locations = pd.concat([
      df[["start_station_id", "start_lat", "start_lng"]].rename(
          columns={"start_station_id": "id", "start_lat": "lat", "start_lng": "long"}
      ),
      df[["end_station_id", "end_lat", "end_lng"]].rename(
          columns={"end_station_id": "id", "end_lat": "lat", "end_lng": "long"}
      )
  ]).dropna(subset=["id"]).drop_duplicates()
  print("station locations length: ", len(station_locations))

  # Take first name and coordinates for each station id in case of conflicting info
  station_names = station_names.groupby("id", as_index=False).first()
  station_locations = station_locations.groupby("id", as_index=False).first()

  # Add station name and location columns to dataframe
  station_counts = (
      station_counts
        .merge(station_names, on="id", how="left")
        .merge(station_locations, on="id", how="left")
  )

  # Turn into geodataframe with crs 4326
  station_counts_gdf = gpd.GeoDataFrame(
    station_counts,
    geometry=gpd.points_from_xy(station_counts["long"], station_counts["lat"]),
    crs="EPSG:4326"
  )

  return station_counts_gdf

In [ ]:
i = 14
f"{(i+1):02}"

In [ ]:
def read_in_data(year_str):
  # year_prefixes = ['2024', '2025', '2026']
  fpath_prefix = '/content/drive/MyDrive/255 Final Project/Data/BikeshareData/'
  fpath_suffix = '-baywheels-tripdata.csv'
  oak_berk_df_list = []

  # for year in year_prefixes:
  for i in range(12):
    if (year_str == '2026' and i > 1):
      break
    # Get file path
    fpath_str = year_str + f"{(i+1):02}"
    file_path = fpath_prefix + fpath_str + fpath_suffix
    print(file_path)

    # Create oakland berkeley dataframe and add to list
    df = create_oak_berk_df(file_path)
    oak_berk_df_list.append(df)

  # Concat all dataframes into one dataframe and return it
  oak_berk_df = pd.concat(oak_berk_df_list)
  return oak_berk_df

# Clean data and export to CSV

In [ ]:
oak_berk_df_2024 = read_in_data('2024')

In [ ]:
oak_berk_df_2025 = read_in_data('2025')

In [ ]:
oak_berk_df_2026 = read_in_data('2026')

In [ ]:
oak_berk_df_2026

In [ ]:
# check station ids are consistently unique with name
pd.set_option('display.max_rows', None)
oak_berk_df_2026.groupby("start_station_name")[["start_station_id"]].nunique()


In [ ]:
oak_berk_df_2026.loc[oak_berk_df_2026['start_station_id'].isna()]

In [ ]:
oak_berk_df_2024.to_csv('/content/drive/MyDrive/255 Final Project/Data/OaklandBerkeleyData2024.csv', index=False)

In [ ]:
oak_berk_df_2025.to_csv('/content/drive/MyDrive/255 Final Project/Data/OaklandBerkeleyData2025.csv', index=False)

In [ ]:
oak_berk_df_2026.to_csv('/content/drive/MyDrive/255 Final Project/Data/OaklandBerkeleyData2026.csv', index=False)

# Create cleaned geodataframes and export

In [ ]:
counts_gdf_24 = create_station_counts_gdf(oak_berk_df_2024)
counts_gdf_24

In [ ]:
df_24 = counts_gdf_24.drop(columns='geometry')
df_24

In [ ]:
counts_gdf_25 = create_station_counts_gdf(oak_berk_df_2025)
counts_gdf_25

In [ ]:
df_25 = counts_gdf_25.drop(columns='geometry')
df_25

In [ ]:
counts_gdf_26 = create_station_counts_gdf(oak_berk_df_2026)
counts_gdf_26

In [ ]:
df_26 = counts_gdf_26.drop(columns='geometry')
df_26

In [ ]:
counts_gdf_24.to_file("/content/drive/MyDrive/255 Final Project/Data/stations_24.geojson", driver="GeoJSON")

In [ ]:
df_24.to_csv("/content/drive/MyDrive/255 Final Project/Data/stations_24.csv", index=False)

In [ ]:
counts_gdf_25.to_file("/content/drive/MyDrive/255 Final Project/Data/stations_25.geojson", driver="GeoJSON")

In [ ]:
df_25.to_csv("/content/drive/MyDrive/255 Final Project/Data/stations_25.csv", index=False)

In [ ]:
counts_gdf_26.to_file("/content/drive/MyDrive/255 Final Project/Data/stations_26.geojson", driver="GeoJSON")

In [ ]:
df_26.to_csv("/content/drive/MyDrive/255 Final Project/Data/stations_26.csv", index=False)

In [ ]:
# counts_gdf_24.to_csv('/content/drive/MyDrive/255 Final Project/Data/StationCountData2024.csv', index=False)

In [ ]:
# counts_gdf_25.to_csv('/content/drive/MyDrive/255 Final Project/Data/StationCountData2025.csv', index=False)

In [ ]:
# counts_gdf_26.to_csv('/content/drive/MyDrive/255 Final Project/Data/StationCountData2026.csv', index=False)